<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=312713200" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model
 
**Compliant with ARC-AGI-2 Competition Rules**
 
This notebook is designed for the ARC-AGI-2 competition on Kaggle. It follows all official requirements:
- Output file: `submission.json` with correct structure (see below)
- No internet access or external dependencies required for submission
- All code is open source and documented
- Each test input gets 2 predictions (attempt_1, attempt_2) as required
 
## Approach
 
* Generate output grids filled with zeros or simple rules (baseline)
* Match input grid dimensions
* Ensure correct submission format
 
## Submission Format Example
```json
{"00576224": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 "009d5c81": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}]}
```
 
## Next Steps
- Add more advanced rule-based or learning-based reasoning
- Modularize code for clarity and reproducibility

In [ ]:
# ARC-AGI-2 Baseline Submission Notebook
# Compliant with competition rules and requirements
import json
import os
import numpy as np
from collections import Counter
 
# Detect rerun mode for Kaggle submission
is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
 
# Load the correct data file based on environment
if is_rerun:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
else:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
 
with open(path) as f:
    data = json.load(f)
 
# --- Utility Functions ---
def copy_grid(g):
    """Return a deep copy of a grid."""
    return [row[:] for row in g]
 
def zero_grid(h, w):
    """Return a grid of size h x w filled with zeros."""
    return [[0]*w for _ in range(h)]
 
def similarity(a, b):
    """Compute similarity between two grids (fraction of matching cells)."""
    if len(a) != len(b) or len(a[0]) != len(b[0]):
        return 0
    same = sum(a[i][j] == b[i][j] for i in range(len(a)) for j in range(len(a[0])))
    return same / (len(a)*len(a[0]))
 
def most_common_color(grid):
    """Return the most common color (integer) in a grid."""
    flat = [c for row in grid for c in row]
    return Counter(flat).most_common(1)[0][0]
 
# --- Transformations ---
def rotate90(g):
    return list(map(list, zip(*g[::-1])))
 
def flip_h(g):
    return [row[::-1] for row in g]
 
def flip_v(g):
    return g[::-1]
 
# --- Scaling & Tiling ---
def apply_scaling(inp, factor):
    arr = np.array(inp)
    return np.repeat(np.repeat(arr, factor, axis=0), factor, axis=1).tolist()
 
def apply_tiling(inp, target_h, target_w):
    arr = np.array(inp)
    reps = (target_h // len(inp), target_w // len(inp[0]))
    return np.tile(arr, reps).tolist()
 
def predict_output_shape(task, inp):
    h, w = len(inp), len(inp[0])
    for pair in task["train"]:
        in_h, in_w = len(pair["input"]), len(pair["input"][0])
        out_h, out_w = len(pair["output"]), len(pair["output"][0])
        if out_h % in_h == 0 and out_w % in_w == 0:
            return out_h, out_w
    return h, w
 
# --- Symmetry ---
def symmetry_complete(g):
    return [row + row[::-1] for row in g]
 
# --- Candidate Generation ---
def generate_candidates(inp, task):
    h, w = len(inp), len(inp[0])
    target_h, target_w = predict_output_shape(task, inp)
    cands = []
    # Basic candidates
    cands.append(copy_grid(inp))
    cands.append(zero_grid(h, w))
    cands.append([[most_common_color(inp)]*w for _ in range(h)])
    # Rotations and flips
    cands.append(rotate90(inp))
    cands.append(flip_h(inp))
    cands.append(flip_v(inp))
    # Scaling
    if target_h > h:
        factor = target_h // h
        cands.append(apply_scaling(inp, factor))
    # Tiling
    if target_h > h:
        cands.append(apply_tiling(inp, target_h, target_w))
    # Symmetry
    cands.append(symmetry_complete(inp))
    return cands
 
# --- Rule Validation ---
def validate_rule(rule_func, task):
    for pair in task["train"]:
        pred = rule_func(pair["input"])
        if pred != pair["output"]:
            return False
    return True
 
# --- Main Solver ---
def solve_task(task):
    results = []
    for test_case in task["test"]:
        inp = test_case["input"]
        cands = generate_candidates(inp, task)
        # Try perfect rule first
        for c in cands:
            if validate_rule(lambda x: c, task):
                best1 = c
                best2 = c
                break
        else:
            # fallback: similarity to last train output
            target = task["train"][-1]["output"] if task["train"] else None
            scored = []
            for c in cands:
                score = similarity(c, target) if target else 0
                scored.append((score, c))
            scored.sort(reverse=True, key=lambda x: x[0])
            best1 = scored[0][1]
            best2 = scored[1][1] if len(scored) > 1 else best1
        results.append({
            "attempt_1": best1,
            "attempt_2": best2
        })
    return results
 
# --- Build Submission ---
submission = {}
for task_id, task in data.items():
    submission[task_id] = solve_task(task)
 
# --- Save Submission ---
with open("submission.json", "w") as f:
    json.dump(submission, f)
 
print("Submission file 'submission.json' created and ready for upload.")

V24 FINAL SUBMISSION READY!


## Notes and Compliance Checklist
- [x] Output file is named `submission.json` and matches required format
- [x] No internet access or external dependencies required for submission
- [x] All code is open source and documented
- [x] Each test input gets 2 predictions (attempt_1, attempt_2)
- [x] Notebook is modular and ready for further improvements
 
**Next steps:**
- Add more advanced rule-based or learning-based reasoning
- Add more comments and docstrings for clarity
- Test on evaluation set before final submission

## Towards a Rule-Based ARC Solver
This section upgrades the baseline to a simple rule-based model. The model will learn basic color mapping rules from the training pairs and apply them to the test input. This is a stepping stone towards a more generalizable model.

In [1]:
# --- Simple Rule-Based Model (Modular, Submission Format) ---
def learn_color_mapping(train_pairs):
    """Learn a mapping from input colors to output colors based on train pairs.
    Args:
        train_pairs (list): List of dicts with 'input' and 'output' grids.
    Returns:
        dict: Mapping from input color to output color."""
    mapping = {}
    for pair in train_pairs:
        inp = pair["input"]
        out = pair["output"]
        for i in range(min(len(inp), len(out))):
            for j in range(min(len(inp[0]), len(out[0]))):
                mapping[inp[i][j]] = out[i][j]
    return mapping
 
def apply_color_mapping(inp, mapping):
    """Apply learned color mapping to input grid.
    Args:
        inp (list): Input grid.
        mapping (dict): Color mapping.
    Returns:
        list: Output grid after mapping."""
    h, w = len(inp), len(inp[0])
    return [[mapping.get(inp[i][j], 0) for j in range(w)] for i in range(h)]
 
def solve_task_rule_based(task):
    """Try to solve using color mapping rule, fallback to baseline if not possible.
    Args:
        task (dict): ARC task with 'train' and 'test'.
    Returns:
        list: List of dicts with 'attempt_1' and 'attempt_2' for each test input, matching submission format."""
    results = []
    mapping = learn_color_mapping(task["train"]) if task["train"] else {}
    for test_case in task["test"]:
        inp = test_case["input"]
        if mapping:
            pred = apply_color_mapping(inp, mapping)
        else:
            pred = zero_grid(len(inp), len(inp[0]))
        # For attempt_2, fallback to most common color
        fallback = [[most_common_color(inp)]*len(inp[0]) for _ in range(len(inp))]
        results.append({
            "attempt_1": pred,
            "attempt_2": fallback
        })
    return results
 
def build_submission(data):
    """Build submission dict for all tasks using rule-based model, matching required format.
    Args:
        data (dict): All ARC tasks.
    Returns:
        dict: Submission dictionary with correct structure."""
    submission = {}
    for task_id, task in data.items():
        submission[task_id] = solve_task_rule_based(task)
    return submission
 
# --- Main: Train and Generate Submission, then Test Format ---
if __name__ == "__main__" or True:
    submission = build_submission(data)
    # Save submission in required format
    with open("submission.json", "w") as f:
        json.dump(submission, f)
    print("Rule-based submission file 'submission.json' created and ready for upload.")
    # --- Test: Check format for a few tasks ---
    for task_id, outputs in list(submission.items())[:2]:
        print(f"Task {task_id} example output:")
        for i, pred in enumerate(outputs):
            print(f"  Test {i+1}: Attempt 1 shape: {len(pred['attempt_1'])}x{len(pred['attempt_1'][0]) if pred['attempt_1'] else 0}, Attempt 2 shape: {len(pred['attempt_2'])}x{len(pred['attempt_2'][0]) if pred['attempt_2'] else 0}")

NameError: name 'data' is not defined

## Next Model Steps: Advanced Reasoning and Evaluation
This section provides a template for adding more advanced reasoning (pattern detection, shape/color rules, etc.) and for evaluating model performance on the validation set.

In [ ]:
# --- Evaluation Utilities ---
def evaluate_submission(submission, solutions):
    """Evaluate submission accuracy against ground truth solutions.
    Args:
        submission (dict): Model predictions.
        solutions (dict): Ground truth outputs.
    Returns:
        float: Accuracy score (0-1)."""
    total, correct = 0, 0
    for task_id, outputs in solutions.items():
        preds = submission.get(task_id, [])
        for i, gt in enumerate(outputs):
            if i >= len(preds):
                continue
            attempts = preds[i]
            if (attempts["attempt_1"] == gt["output"]) or (attempts["attempt_2"] == gt["output"]):
                correct += 1
            total += 1
    return correct / total if total else 0
 
# Example usage (uncomment and set correct path to use):
# with open('arc-agi_evaluation_solutions.json') as f:
#     solutions = json.load(f)
# with open('submission.json') as f:
#     submission = json.load(f)
# acc = evaluate_submission(submission, solutions)
# print(f'Validation accuracy: {acc:.3f}')